In [1]:
import numpy as np
import pandas as pd
import lightning.pytorch as pl
import torch
import time

from pytorch_forecasting import Baseline, TemporalFusionTransformer
from pytorch_forecasting.data import (  # GroupNormalizer,
    NaNLabelEncoder,
    TimeSeriesDataSet,
)
from pytorch_forecasting.metrics import MAE, MAPE, RMSE, SMAPE, QuantileLoss
from pytorch_forecasting.models.temporal_fusion_transformer.tuning import (
    optimize_hyperparameters,
)
from lightning.pytorch.callbacks import (
    EarlyStopping,
    LearningRateMonitor,
    ModelCheckpoint,
)
from lightning.pytorch.loggers import TensorBoardLogger

from snowflake_data_loader import snowflakeDataLoader

In [2]:
data = pd.read_csv('../data/mam_data_snowpark_poc_small.csv')

In [3]:
snowflake = snowflakeDataLoader(
        data=data,
        min_dates=15,
        top_k = 1000
    )

snowflake(
        min_prediction_length=3,
        max_prediction_length=10,
        min_encoder_length=4,
        max_encoder_length=256,
        batch_size=64,
        num_workers=2
    )


using 2 ASINs
(2835, 42)
1891
training_cutoff=1881
Time series params:  {'time_idx': 'time_idx', 'target': 'order_revenue', 'group_ids': ['amazon_id'], 'weight': None, 'max_encoder_length': 256, 'min_encoder_length': 4, 'min_prediction_idx': 0, 'min_prediction_length': 3, 'max_prediction_length': 10, 'static_categoricals': ['amazon_id', 'pepsi_brand', 'business_unit', 'sub_business_unit', 'category', 'sub_category'], 'static_reals': ['encoder_length'], 'time_varying_known_categoricals': ['holiday', 'quarter', 'month', 'day', 'weekend'], 'time_varying_known_reals': ['search_spend', 'search_spend_past', 'relative_time_idx'], 'time_varying_unknown_categoricals': [], 'time_varying_unknown_reals': ['online_units', 'average_sales_price', 'online_units_past', 'average_sales_price_past', 'price_index', 'rep_oos', 'rep_oos_past', 'order_revenue_past'], 'variable_groups': {}, 'constant_fill_strategy': {}, 'allow_missing_timesteps': True, 'lags': {}, 'add_relative_time_idx': True, 'add_target_sca

In [4]:
early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=10, verbose=False, mode="min")
lr_logger = LearningRateMonitor()
run_id = str(time.time()).replace(".", "")
dir = '/tmp/'
logger = TensorBoardLogger(save_dir=f"{dir}/logs")
checkpoint = ModelCheckpoint(
    dirpath=dir, 
    filename="{epoch}-{val_loss:.2f}-{other_metric:.2f}",
    monitor="val_loss"
)

In [13]:
trainer = pl.Trainer(
    max_epochs=1,
    gradient_clip_val=0.5,
    callbacks=[early_stop_callback, lr_logger],
    logger=logger,
    default_root_dir=f"{dir}/data/",
    limit_train_batches=10,
    limit_val_batches=10,
)

tft = TemporalFusionTransformer.from_dataset(
    snowflake.dataset,
    learning_rate=0.005,
    hidden_size=78,
    attention_head_size=12,
    dropout=0.35,
    hidden_continuous_size=9,
    output_size=7,
    loss=QuantileLoss(),
    logging_metrics=[SMAPE(), MAPE(), RMSE(), MAE()],
    reduce_on_plateau_patience=4,
)

print(snowflake.train_dataloader)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs


In [14]:
%%time
trainer.fit(
    tft,
    snowflake.train_dataloader,
    snowflake.val_dataloader,
)


   | Name                               | Type                            | Params
----------------------------------------------------------------------------------------
0  | loss                               | QuantileLoss                    | 0     
1  | logging_metrics                    | ModuleList                      | 0     
2  | input_embeddings                   | MultiEmbedding                  | 440   
3  | prescalers                         | ModuleDict                      | 216   
4  | static_variable_selection          | VariableSelectionNetwork        | 3.9 K 
5  | encoder_variable_selection         | VariableSelectionNetwork        | 28.9 K
6  | decoder_variable_selection         | VariableSelectionNetwork        | 8.9 K 
7  | static_context_variable_selection  | GatedResidualNetwork            | 24.8 K
8  | static_context_initial_hidden_lstm | GatedResidualNetwork            | 24.8 K
9  | static_context_initial_cell_lstm   | GatedResidualNetwork            | 24.8

Sanity Checking: 0it [00:00, ?it/s]

/Users/jprusa/miniconda3/envs/snowpark/lib/python3.8/site-packages/lightning/pytorch/loops/fit_loop.py:280: PossibleUserWarning: The number of training batches (10) is smaller than the logging interval Trainer(log_every_n_steps=50). Set a lower value for log_every_n_steps if you want to see logs for the training epoch.
  rank_zero_warn(


Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

`Trainer.fit` stopped: `max_epochs=1` reached.


CPU times: user 20 s, sys: 1.86 s, total: 21.8 s
Wall time: 44 s


In [16]:
# 10 minibatches
# 2.43 sec/minibatch
# 24 for all 10
# 44 including startup (sanity checks etc.)
# 20 sec startup


# 1 epoch
# 1.29 sec/minibatch
# 1:10 per epoch including validation
# 1:33 including startup (sanity checks etc.)
# 23 sec of startup

# 5 epoch
# sec per minibatch and epoch length look to be very similar
# 6:06 for 5 epochs
# 16ish seconds of startup assuming 1:10 per epoch
# basically the same


# training with sproc

# 10 epochs 5.5x slower
# 1st epoch

In [ ]:
# actual model training for sproc and local
# estimate for full 
# estimate for hpo

In [19]:
70*100/60/60+

1.9444444444444444